WARNING: Notebook was written during development. May not work as intended. Retained here only for completion and future reference.

# Magnetic actuation of an annular hard-magnetic rod

Forward analysis only, for the method figure: the same annular rod as `annular_swell`
(same length, radii and sector tiling), but every sector now carries a remnant
magnetisation of fixed magnitude whose *direction* is a continuous function of arc
length `s`, radial position `r` and polar angle `θ`. A small library of closed-form
magnetisation layouts is actuated by a uniform applied field, and the reference layout
is shown beside the actuated shape.

In [ ]:
import pathlib

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Wedge

import crops.src.bc as _bc
import crops.src.fe_cosserat as _fe
import crops.src.mesher as _mesher
import crops.src.solver as _solver
import crops.src.utils as _utils
import crops.src.viz as _viz

jax.config.update("jax_enable_x64", True)
plt.rcParams.update(_viz.default_plot_settings)


figure_dir = pathlib.Path("crops/notebooks/dev/paper/figures")
figure_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
background_color = "#FFFFFF"
plt.rcParams.update(
  {
    "figure.facecolor": background_color,
    "axes.facecolor": background_color,
    "savefig.facecolor": background_color,
    "axes.labelsize": 28,
    "axes.titlesize": 28,
    "xtick.labelsize": 28,
    "ytick.labelsize": 28,
    "legend.fontsize": 28,
  }
)

### Annular cross-section

The wall between `inner_radius` and `outer_radius` is split into `num_rings` rings of
`num_sectors` sectors each. Every sector is one fiber of the rod model: its centroid is
the quadrature point and its area the quadrature weight, so the resolution of the
material layout in the cross-section is set here.

In [ ]:
rod_length = 1.0
num_elems = 120
num_nodes = num_elems + 1
parametric_coords = jnp.linspace(0.0, rod_length, num_nodes)

outer_radius = rod_length / 30.0
inner_radius = 0.5 * outer_radius
num_rings = 4
num_sectors = 16
num_fibers = num_rings * num_sectors

# Sector layout, ring-major from the inner wall outwards, counter-clockwise from d1.
ring_edges = np.linspace(inner_radius, outer_radius, num_rings + 1)
sector_edges = np.linspace(0.0, 2.0 * np.pi, num_sectors + 1)
ring_index, sector_index = np.divmod(np.arange(num_fibers), num_sectors)
sector_inner = ring_edges[ring_index]
sector_outer = ring_edges[ring_index + 1]
sector_start = sector_edges[sector_index]
sector_end = sector_edges[sector_index + 1]

# Closed-form area, centroid and centroidal second moments of an annular sector.
span = sector_end - sector_start
sector_areas = 0.5 * span * (sector_outer**2 - sector_inner**2)
cubic = (sector_outer**3 - sector_inner**3) / 3.0
centroid_d1 = cubic * (np.sin(sector_end) - np.sin(sector_start)) / sector_areas
centroid_d2 = cubic * (np.cos(sector_start) - np.cos(sector_end)) / sector_areas
quartic = (sector_outer**4 - sector_inner**4) / 4.0
double_angle = 0.25 * (np.sin(2.0 * sector_end) - np.sin(2.0 * sector_start))
moi_about_d1 = quartic * (0.5 * span - double_angle) - sector_areas * centroid_d2**2
moi_about_d2 = quartic * (0.5 * span + double_angle) - sector_areas * centroid_d1**2


def along_rod(values):
  """Repeat a per-fiber quantity over every element."""
  return np.broadcast_to(values, (num_elems,) + np.shape(values))


rod = _mesher.build_rod_geometry(
  analytical_curve=lambda s: jnp.array([s, 0.0, 0.0]),
  parametric_coords=parametric_coords,
  fiber_offsets=jnp.asarray(along_rod(np.stack([centroid_d1, centroid_d2], axis=-1))),
  fiber_areas=jnp.asarray(along_rod(sector_areas)),
  fiber_mass_density=jnp.full((num_elems, num_fibers), 1100.0),
  fiber_centroid_moi=jnp.asarray(along_rod(np.stack([moi_about_d1, moi_about_d2], -1))),
  seed_binormal=jnp.array([0.0, 0.0, 1.0]),
)

# Sector geometry in the layout `_viz.plot_rod_annular_sections` expects.
section = dict(
  fiber_inner_radii=along_rod(sector_inner),
  fiber_outer_radii=along_rod(sector_outer),
  fiber_start_angles=along_rod(sector_start),
  fiber_end_angles=along_rod(sector_end),
)

# Normalised coordinates of every element-fiber cell for the layout functions:
# s in (0, 1) along the rod, r in (0, 1) across the wall, theta in [0, 2pi) from d1.
s_hat = np.asarray(0.5 * (parametric_coords[:-1] + parametric_coords[1:])) / rod_length
r_hat = (np.hypot(centroid_d1, centroid_d2) - inner_radius) / (
  outer_radius - inner_radius
)
theta = np.arctan2(centroid_d2, centroid_d1) % (2.0 * np.pi)
S, R, TH = np.broadcast_arrays(s_hat[:, None], r_hat[None, :], theta[None, :])

print(f"{num_fibers} sectors per section, {num_elems * num_fibers} cells")
print(f"wall / outer radius = {(outer_radius - inner_radius) / outer_radius:.2f}")
print(f"length / outer diameter = {rod_length / (2 * outer_radius):.0f}")

### Magnetisation layouts

Each layout returns a unit magnetisation direction `m` on the `(num_elems, num_fibers)`
cells, resolved in the material frame `{d1, d2, d3}`. `from_angles` builds it from an
azimuth in the cross-section (from `d1` towards `d2`, like `θ`) and an elevation out of
the section towards the tangent `d3`, so `+π/2` is "along the rod, towards the tip".
`only_where` zeroes `m` on cells that stay passive elastomer. Every design also carries
its applied flux density `B` as a vector in the global frame, where the reference rod
lies along `x` with `d1 = y` and `d2 = z`; `field_at(angle)` gives a unit field in the
`x`-`y` plane at `angle` from the rod axis towards `d1`.

The couple per unit volume is `m × B`, so only the part of `m` perpendicular to the
field acts, a section is torqued about the axis perpendicular to both, and it stops
turning once `m` has swung into the field. That last point shapes the library: unlike
swelling, no section can turn further than the angle between its `m` and `B`, and a
section magnetised against the field is unstable and flips.

- **Transverse field (along `d1`)**: the `d3` component bends the rod in the `d1`-`d3`
  plane towards the field, alternating its sign alternates the curvature; the `d2`
  component twists it about its axis; the `d1` component is inert.
- **Axial field (along `d3`)**: the in-plane component bends the rod about the in-plane
  axis perpendicular to it, so rotating the azimuth along `s` rotates the bending plane.
- **Oblique field**: pointing the field partly back along the rod lets an axially
  magnetised tip swing through more than a right angle, giving hooks and deep arcs.

The couple depends on the section-integrated magnetisation, so patterns that vary
across the section (radial, circumferential) largely cancel; the layouts below vary
along `s`, and `r`, `θ` are kept for masks such as a magnetised outer shell.

Layouts compose with plain numpy on the `(num_elems, num_fibers, 3)` arrays.

In [ ]:
def from_angles(azimuth, elevation):
  """Unit magnetisation direction on every cell from two angles in the material frame.

  `azimuth` is measured in the cross-section from `d1` towards `d2` (like `theta`) and
  `elevation` out of the cross-section towards the tangent `d3`, so `+pi/2` points along
  the rod towards the tip and `0` lies in the section. An elevation beyond `+-pi/2`
  keeps rotating through the plane, as a vector should. Scalars broadcast over cells.
  """
  azimuth, elevation = (np.broadcast_to(a, S.shape) for a in (azimuth, elevation))
  return np.stack(
    [
      np.cos(elevation) * np.cos(azimuth),
      np.cos(elevation) * np.sin(azimuth),
      np.sin(elevation),
    ],
    axis=-1,
  )


def only_where(magnetization, keep):
  """Zero the magnetisation (passive elastomer) on cells where `keep` is False."""
  return np.where(np.asarray(keep)[..., None], magnetization, 0.0)


def field_at(angle_deg):
  """Unit applied field in the x-y plane, `angle_deg` from the rod axis towards d1."""
  angle = np.radians(angle_deg)
  return np.array([np.cos(angle), np.sin(angle), 0.0])


def axial(s, r, theta):
  """Uniform magnetisation along the rod. In a transverse field every section is
  torqued the same way, so the rod bends towards the field like a cantilever under a
  distributed couple; with the field pointing partly back along the rod the tip swings
  past the vertical and the rod hooks.
  """
  return from_angles(azimuth=0.0, elevation=np.pi / 2)


def tip_magnet(s, r, theta, magnetised_fraction=0.25):
  """Axial magnetisation on the last `magnetised_fraction` of the rod only. The couple
  acts at the tip, so the passive part carries a uniform moment and bends into a
  circular arc: the magnetic counterpart of the swelling bilayer.
  """
  return only_where(axial(s, r, theta), s > 1.0 - magnetised_fraction)


def undulation(s, r, theta, num_waves=1.0):
  """Magnetisation swinging smoothly between `+d3` and `-d3` through `d1`,
  `num_waves` times along the rod. In a transverse field along `d1` the curvature
  alternates sign with the axial component, so the rod takes a sinusoidal wave; the
  swing always passes through the field direction, never against it, so the wave
  survives strong fields instead of flipping into a hook.
  """
  return from_angles(
    azimuth=0.0, elevation=0.5 * np.pi * np.cos(2.0 * np.pi * num_waves * s)
  )


def serpentine(s, r, theta, num_segments=3):
  """Axial magnetisation that flips sign at every one of `num_segments` equal
  segments: the two-phase version of `undulation`.
  """
  sign = 1.0 - 2.0 * (np.floor(num_segments * s) % 2)
  return from_angles(azimuth=0.0, elevation=sign * np.pi / 2)


def coil(s, r, theta, num_turns=1.0):
  """Transverse magnetisation whose azimuth winds `num_turns` times around the rod. In
  an axial field every section is torqued about the in-plane axis perpendicular to its
  magnetisation, so the bending plane rotates along the rod and the shape winds out of
  plane.
  """
  return from_angles(azimuth=2.0 * np.pi * num_turns * s, elevation=0.0)


def rotating_segments(s, r, theta, num_segments=3, angle_step=np.pi / 2):
  """Transverse magnetisation whose azimuth jumps by `angle_step` at every one of
  `num_segments` equal segments; `pi / 2` in an axial field gives a 3D zigzag.
  """
  return from_angles(azimuth=angle_step * np.floor(num_segments * s), elevation=0.0)


# Candidate layouts as (magnetisation, applied flux density [T]). Add or edit entries
# here, then pick one below.
designs = {
  "hook": (lambda: axial(S, R, TH), 0.025 * field_at(135.0)),
  "tip_magnet": (lambda: tip_magnet(S, R, TH), 0.075 * field_at(150.0)),
  "undulation": (lambda: undulation(S, R, TH, num_waves=1.0), 0.045 * field_at(90.0)),
  "serpentine": (lambda: serpentine(S, R, TH, num_segments=3), 0.045 * field_at(90.0)),
  "coil": (lambda: coil(S, R, TH, num_turns=1.0), 0.05 * field_at(0.0)),
  "zigzag": (lambda: rotating_segments(S, R, TH, num_segments=3), 0.05 * field_at(0.0)),
  # Outer-wall-only coil: magnetised only on the outer half of the wall, so it needs
  # twice the field for the same couple.
  "shell_coil": (lambda: only_where(coil(S, R, TH), R > 0.5), 0.1 * field_at(0.0)),
}

### Colour sphere and cross-section preview

A magnetisation direction is a point on the unit sphere, so the sphere itself is the
legend: every direction gets its own colour and no arrows are needed. `direction_to_rgb`
builds it in Oklch, which is perceptually even, from the material-frame components.
Every colour sphere has two achromatic poles, where the hue is undefined, so they are
put on `d2`, the out-of-plane direction that layouts rarely use: the angle in the
`d1`-`d3` bending plane (from `d3` towards `d1`) sets the hue, so axial and in-plane
magnetisations all get vivid colours; the component along `d2` sets the lightness
(light towards `-d2`, dark towards `+d2`); and the magnitude in the bending plane sets
the chroma, so every meridian fades evenly into the poles. The map is linear in the
vector, so a passive cell (zero vector) is the mid grey at the centre of the sphere.

The sphere is drawn once as its own figure, oriented like the reference rod
(`d3` along `x`, `d1` along `y`, `d2` along `z`), next to its flat azimuth-elevation
map. Then the end-on sections of every layout at a few stations, as wedges in the
`d1`-`d2` plane, to check a layout before solving.

In [ ]:
def direction_to_rgb(m, lightness=(0.9, 0.45), chroma=0.22, hue_offset=0.5):
  """Colour sphere with its poles on d2: hue from the angle of `m` in the d3-d1
  bending plane (from d3 towards d1), lightness from its d2 component
  (`lightness[0]` at -d2, `lightness[1]` at +d2) and chroma from its in-plane
  magnitude, in Oklch (Ottosson 2020), clipped to sRGB. Broadcasts over `(..., 3)`;
  a zero vector gives the mid grey at the centre of the sphere.
  """
  m = np.asarray(m, dtype=float)
  hue = np.arctan2(m[..., 0], m[..., 2]) + hue_offset
  a, b = chroma * np.hypot(m[..., 2], m[..., 0]) * np.stack([np.cos(hue), np.sin(hue)])
  lo, hi = lightness
  ell = 0.5 * (lo + hi) + 0.5 * (hi - lo) * m[..., 1]
  lms = (
    np.stack(
      [
        ell + 0.3963377774 * a + 0.2158037573 * b,
        ell - 0.1055613458 * a - 0.0638541728 * b,
        ell - 0.0894841775 * a - 1.2914855480 * b,
      ],
      axis=-1,
    )
    ** 3
  )
  lms_to_rgb = np.array(
    [
      [4.0767416621, -3.3077115913, 0.2309699292],
      [-1.2684380046, 2.6097574011, -0.3413193965],
      [-0.0041960863, -0.7034186147, 1.7076147010],
    ]
  )
  linear = np.clip(lms @ lms_to_rgb.T, 0.0, 1.0)
  return np.where(
    linear <= 0.0031308, 12.92 * linear, 1.055 * linear ** (1 / 2.4) - 0.055
  )


def sphere_directions(resolution):
  """Material-frame directions on a grid of the sphere's own angles: the angle in
  the d3-d1 plane from d3 towards d1, and the elevation towards d2."""
  angle, elevation = np.meshgrid(
    np.linspace(0.0, 2.0 * np.pi, 2 * resolution),
    np.linspace(-np.pi / 2, np.pi / 2, resolution),
  )
  return np.stack(
    [
      np.cos(elevation) * np.sin(angle),
      np.sin(elevation),
      np.cos(elevation) * np.cos(angle),
    ],
    axis=-1,
  )


def draw_color_sphere(ax, view=(25.0, -60.0), resolution=60):
  """Draw the colour sphere on a 3D axis in the material frame, oriented like the
  reference rod (d3 along x, d1 along y, d2 along z), with the three directors marked.
  """
  m = sphere_directions(resolution)
  ax.plot_surface(
    m[..., 2],
    m[..., 0],
    m[..., 1],
    facecolors=direction_to_rgb(m),
    rstride=1,
    cstride=1,
    shade=False,
    linewidth=0.0,
    antialiased=False,
  )
  for axis, label in ((1, "$d_1$"), (2, "$d_2$"), (0, "$d_3$")):
    tick = np.zeros((2, 3))
    tick[:, axis] = [1.0, 1.3]
    ax.plot(*tick.T, color="k", linewidth=1.2)
    ax.text(*(1.5 * tick[1] / 1.3), label, ha="center", va="center", fontsize=11)
  ax.set_xlim3d(-1.5, 1.5)
  ax.set_ylim3d(-1.5, 1.5)
  ax.set_zlim3d(-1.5, 1.5)
  ax.set_box_aspect((1.0, 1.0, 1.0), zoom=1.5)
  ax.set_axis_off()
  ax.view_init(elev=view[0], azim=view[1])


def draw_color_map(ax, resolution=180):
  """Draw the colour sphere unrolled as an angle-elevation map on a 2D axis."""
  ax.imshow(
    direction_to_rgb(sphere_directions(resolution)),
    extent=(0.0, 360.0, 00.0, 180.0),
    origin="lower",
    aspect=1.0,
  )
  ax.set_xticks([0,  180, 360])
  ax.set_yticks([0, 90, 180])
  ax.set_xlabel(r"$\alpha$ [deg]")
  ax.set_ylabel(r"$\beta$ [deg]")


fig = plt.figure(figsize=(8, 3.4))
# draw_color_sphere(fig.add_subplot(1, 2, 1, projection="3d"))
draw_color_map(fig.add_subplot(1, 1, 1))
fig.tight_layout()
fig.savefig(figure_dir / "annular_magnetic_color_sphere.pdf", dpi=600)
plt.show()

In [ ]:
def draw_section(ax, magnetization_at_elem, title=None):
  """Draw one cross-section as annular-sector wedges coloured by the colour sphere."""
  colors = direction_to_rgb(magnetization_at_elem)
  for f in range(num_fibers):
    ax.add_patch(
      Wedge(
        center=(0.0, 0.0),
        r=sector_outer[f],
        theta1=np.degrees(sector_start[f]),
        theta2=np.degrees(sector_end[f]),
        width=sector_outer[f] - sector_inner[f],
        facecolor=colors[f],
        edgecolor="k",
        linewidth=0.2,
      )
    )
  ax.set_xlim(-1.05 * outer_radius, 1.05 * outer_radius)
  ax.set_ylim(-1.05 * outer_radius, 1.05 * outer_radius)
  ax.set_aspect("equal")
  ax.set_axis_off()
  if title is not None:
    ax.set_title(title, fontsize=9)


def preview_sections(magnetization, stations=(0.05, 0.3, 0.5, 0.7, 0.95), title=None):
  """A row of cross-sections at the given fractions of the rod length."""
  fig, axes = plt.subplots(1, len(stations), figsize=(2.2 * len(stations), 2.4))
  for ax, station in zip(axes, stations):
    elem = min(int(station * num_elems), num_elems - 1)
    draw_section(ax, magnetization[elem], title=f"s/L = {station:.2f}")
  if title is not None:
    fig.suptitle(title, fontsize=10)
  fig.tight_layout()
  return fig


for name, (layout, _) in designs.items():
  preview_sections(layout(), title=name)
plt.show()

### Material, field and boundary conditions

The whole rod is one hard-magnetic elastomer of shear modulus `shear_modulus` with a
remnant flux density of magnitude `remnant_flux` in every magnetised cell. The applied
flux density each design names is ramped linearly from zero over the load steps. The
root is clamped and the rest of the rod is free to move in three dimensions.

The solver wants the remnant flux in the reference global frame, so the material-frame
layout is rotated through the reference element frames before it is passed in.

In [ ]:
# The load is set by the magnetic couple per unit length, A B_r B_a / mu_0, against
# the bending stiffness over the length, E I / L^2. For this tube and material the
# ratio is about 250 B_a / T, so ~25 mT turns the hook through a right angle; the
# actual turn is less because the couple fades as each section aligns with the field.
shear_modulus = 3.0e5  # G [Pa]
remnant_flux = 0.1  # B_r [T]

num_steps = 20
ramp = np.linspace(0.0, 1.0, num_steps + 1)[1:]

clamped_dofs = (
  _utils.Translation.X,
  _utils.Translation.Y,
  _utils.Translation.Z,
  _utils.Rotation.THETA_X,
  _utils.Rotation.THETA_Y,
  _utils.Rotation.THETA_Z,
)
boundary_list = [
  _bc.DirichletBC(
    values=(dof, [np.array([0.0]) for _ in range(num_steps)]),
    node_targets=[np.array([0]) for _ in range(num_steps)],
    name=f"clamp_{dof}",
  )
  for dof in clamped_dofs
]
bc = _bc.process_boundary_conditions(rod, boundary_list)

nr_solver = _solver.NewtonRaphson(
  solver_type=_solver.NonlinearSolvers.ARMIJO_BACKTRACKING_NEWTON_RAPHSON,
  nonlinear_settings={
    "max_iter": 100,
    "threshold": 1e-6,
    "abs_threshold": 1e-6,
    "lam_min": 0.01,
    "line_search_max_iter": 12,
    "line_search_shrink": 0.5,
    "line_search_alpha_min": 1e-6,
    "line_search_armijo_c": 1e-4,
  },
  linear_settings={
    "solver": _solver.LinearSolvers.SCIPY_SPARSE,
    "rtol": 1e-8,
  },
)
problem = _fe.CosseratProblem(rod=rod, bc=bc, nr_solver=nr_solver)

# Reference element frames, rows d1, d2, d3, used to rotate a material-frame
# magnetisation into the global frame the solver expects.
elem_frames_0 = 0.5 * np.asarray(rod.directors_0[:-1] + rod.directors_0[1:])


def solve_layout(magnetization, applied_field):
  """Actuate the rod for a magnetisation direction field and a uniform applied flux
  density [T] and return the deformed centreline coordinates and directors."""
  remnant_mag_flux = remnant_flux * np.einsum(
    "efk, ekd -> efd", magnetization, elem_frames_0
  )
  applied_mag_flux = np.outer(ramp, applied_field)
  u_star, _ = problem.solve(
    u0=jnp.zeros(rod.num_dofs),
    shear_moduli=jnp.full((num_elems, num_fibers), shear_modulus),
    remnant_mag_flux=jnp.asarray(remnant_mag_flux),
    applied_mag_flux=jnp.asarray(applied_mag_flux),
  )
  coords, directors, _ = _mesher.extract_coords_and_director_from_deformation(
    rod, u_star
  )
  return np.asarray(coords), np.asarray(directors)

### Gallery

Actuate every candidate layout and show the reference rod beside the actuated one,
both coloured by the colour sphere in the material frame; the applied field is the
labelled arrow.

In [ ]:
def draw_rod(
  ax,
  coords,
  directors,
  magnetization,
  applied_field=None,
  field_corner=(-0.9, -1.1, 1.1),
  view=(25.0, -60.0),
  num_arc_segments=1,
):
  """Render the annular rod on a 3D axis coloured by the colour sphere of its
  material-frame magnetisation and, optionally, draw the applied field at
  `field_corner`, in half-spans of the bounding box from its centre (so beyond +-1
  sits just outside the box).

  The viewport is fitted to the rod's own bounding box at equal aspect, with slender
  directions held open to a fraction of the longest one, rather than to a cube.
  """
  _viz.plot_rod_annular_sections(
    node_coords=coords,
    directors=directors,
    **section,
    num_arc_segments=num_arc_segments,
    fiber_field=direction_to_rgb(magnetization),
    edge_width=0.1,
    ax=ax,
  )
  low = coords.min(axis=0) - 1.5 * outer_radius
  high = coords.max(axis=0) + 1.5 * outer_radius
  span = np.maximum(high - low, 0.25 * np.max(high - low))
  mid = 0.5 * (low + high)
  if applied_field is not None:
    anchor = mid + 0.5 * span * np.asarray(field_corner)
    field_length = 0.15 * np.max(span)
    direction = applied_field / np.linalg.norm(applied_field)
    ax.quiver(
      *anchor,
      *direction,
      length=field_length,
      normalize=True,
      color="k",
      linewidth=2.0,
      arrow_length_ratio=0.3,
    )
    label_at = anchor + 0.5 * field_length * direction + [0.0, 0.0, 0.06 * np.max(span)]
    ax.text(*label_at, r"$\mathbf{B}$", fontsize=11, ha="center")

  ax.set_xlim3d(mid[0] - 0.5 * span[0], mid[0] + 0.5 * span[0])
  ax.set_ylim3d(mid[1] - 0.5 * span[1], mid[1] + 0.5 * span[1])
  ax.set_zlim3d(mid[2] - 0.5 * span[2], mid[2] + 0.5 * span[2])
  ax.set_box_aspect(span)
  ax.view_init(elev=view[0], azim=view[1])

In [ ]:
results = {}
for name, (layout, applied_field) in designs.items():
  magnetization = layout()
  coords, directors = solve_layout(magnetization, applied_field)
  results[name] = (magnetization, applied_field, coords, directors)

  kinematics = _mesher.compute_discrete_cosserat_kinematics(
    node_coords=jnp.asarray(coords),
    directors=jnp.asarray(directors),
    arc_coords=rod.arc_coords,
  )
  max_curvature = np.abs(np.asarray(kinematics.curvature)).max(axis=0) * rod_length
  chords = np.linalg.norm(np.diff(coords, axis=0), axis=1)
  print(
    f"{name:12s} B = {1e3 * np.linalg.norm(applied_field):4.0f} mT   "
    f"arc length / L = {chords.sum() / rod_length:.3f}   "
    f"max |kappa| L = {np.array2string(max_curvature, precision=2)}"
  )

  fig = plt.figure(figsize=(11, 4))
  ax_ref = fig.add_subplot(1, 2, 1, projection="3d")
  draw_rod(
    ax_ref,
    np.asarray(rod.node_coords),
    np.asarray(rod.directors_0),
    magnetization,
    applied_field=applied_field,
  )
  ax_ref.set_title(f"{name}: reference", fontsize=9)
  ax_def = fig.add_subplot(1, 2, 2, projection="3d")
  draw_rod(ax_def, coords, directors, magnetization, applied_field=applied_field)
  ax_def.set_title(f"{name}: actuated", fontsize=9)
  fig.tight_layout()
  plt.show()

### Method figure

Pick a layout and render it for the paper: the actuated shape with its magnetisation
arrows and the applied field. Arc segments are refined here for smoother rings; the
file is saved as PNG because the polygon count makes a vector file heavy.

In [ ]:
import matplotlib
%matplotlib qt5
design_name = "undulation"
view = (25.0, -60.0)  # (elevation, azimuth); use (90, -90) for the x-y plane

magnetization, applied_field, coords, directors = results[design_name]

fig = plt.figure(figsize=(12, 5))
ax_def = fig.add_subplot(1, 1, 1, projection="3d")
draw_rod(
  ax_def,
  coords,
  directors,
  magnetization,
  applied_field=applied_field,
  view=view,
  num_arc_segments=2,
)
fig.tight_layout()
fig.savefig(figure_dir / f"annular_magnetic_{design_name}.png", dpi=300)
plt.show()